## Final Report: Voting Demographics
Team Members: Ashanti Hatchett, Srinath Pathangae, Kumhyun Song

### Introduction

This project explores how the demographics of voters affect their participation in elections, using voter turnout data from the 2024 U.S. presidential election. The policy problem we address is unequal voter participation across demographic groups, which can undermine democratic representation.

Understanding which demographic groups are more or less likely to vote is important for policymakers and election officials seeking to increase overall voter turnout. If certain groups consistently participate at lower rates, their preferences may be underrepresented in electoral outcomes.

Using the results of our analysis, we aim to predict whether an individual voter is likely to vote. Such predictions could help inform targeted outreach strategies designed to encourage participation among voters who are less likely to vote. In this context, identifying individuals who ultimately do not vote is particularly important, as policy interventions would be directed toward these groups.

To make these predictions, we employ three models: logistic regression, random forests, and neural networks. We evaluate their performance using the F1 score, which is appropriate for our analysis because the dataset is imbalanced, as discussed in the Data section. We define not_voted as 1 and voted as 0, as our analysis focuses on identifying individuals who do not vote.

### Data

Our data source is the IPUMS CPS (Current Population Survey) November 2024 Voting Supplement (<https://cps.ipums.org/>). The dataset includes information on whether individuals voted in the 2024 U.S. presidential election, along with a wide range of demographic characteristics.
From the available variables, we selected 12 features based on their expected relationship with voting behavior:

- Sex (categorical): Sex is a standard demographic factor that may influence voting behavior. We included it because many previous studies suggest differences in political participation across sexes. In our dataset, males exhibit a slightly higher non-voting rate than females.

- Race (categorical, manipulated): Race is another important demographic variable affecting voting behavior. We grouped all smaller or mixed racial categories into an “other” category to reduce sparsity, as these categories contained relatively few observations. Our data show that individuals identified as American Indian or Eskimo have the highest non-voting rates.

![data1.png](plots/data1.png)

- Education (categorical, manipulated): Education level can influence political engagement and interest. Although more detailed subgroups were initially available, we used a simplified classification by grouping education into four main categories: below high school, high school graduate, college graduate, and master’s degree or higher. However, after data cleaning, no individuals remained in the below high school category. Individuals who attended college but did not graduate were grouped with high school graduates. Our results show that high school graduates have the highest non-voting rates.

- Employment Status (categorical, manipulated): Employment status may affect voting behavior through differences in time availability and political engagement. Although subgroups were initially available, we used the broader classification by consolidating them into four main categories: employed, unemployed, retired, and not in the labor force. Our analysis shows that unemployed and not-in-labor-force individuals tend to have higher non-voting rates.

![data2.png](plots/data2.png)

- Nativity (categorical, manipulated): Nativity indicates whether an individual is native-born or foreign-born. We simplified the variable by grouping individuals with at least one native-born parent as “native-born,” and others as “foreign-born.” This reflects differences in familiarity and engagement with U.S. political systems. Foreign-born individuals appear to have higher non-voting rates.

- Region (categorical, manipulated): Political behavior often varies by geographic region. We aggregated subregions into broader categories (Northeast, Midwest, South, and West). The Midwest and South show slightly higher non-voting rates, though differences are relatively modest.

![data3.png](plots/data3.png)

- Mobility Disability (categorical): Mobility limitations may restrict physical access to polling locations. Individuals with mobility-related disabilities show higher non-voting rates in our data.

- Metropolitan (categorical, manipulated): Access to polling locations may differ between metropolitan and non-metropolitan areas. We reduced this variable to a binary indicator (metropolitan vs. non-metropolitan). Individuals living in non-metropolitan areas tend to have slightly higher non-voting rates.

![data4.png](plots/data4.png)

- Marital Status (categorical, manipulated): Marital status may influence voting through social interaction and shared decision-making. We simplified this variable into two categories: with spouse and without spouse. Individuals without a spouse tend to have higher non-voting rates.

![data5.png](plots/data5.png)

- Age (numerical, manipulated): Age is a key predictor of voting behavior. In the dataset, individuals aged 80–84 are recorded as 80, and those aged 85+ are recorded as 85. To address this limitation, we grouped age into five-year intervals and used the midpoint of each interval. Younger individuals show significantly higher non-voting rates.

![data6.png](plots/data6.png)

- Number of Children (numerical): Having children may limit time available for voting. Our dataset indicates that households with more children tend to have higher non-voting rates.

![data7.png](plots/data7.png)

- Income Per Person (numerical, extracted): We created this feature by dividing family income by family size, as family income alone does not account for household size. Since family income was provided in ranges rather than exact values, we used the midpoint of each range as an estimate. This provides a more accurate measure of economic resources per individual. Lower income per person is associated with higher non-voting rates.

![data8.png](plots/data8.png)

We cleaned the dataset by removing observations with missing values. If more than one feature was missing (marked as “not in universe”), the entire row was dropped. After cleaning, the dataset contained 62,327 observations.

We then performed feature engineering and transformations as described above, including grouping categories, creating new variables, and simplifying complex variables. After that, we encoded categorical variables into numerical representations for model input using OneHotEncoder, and normalized numerical variables where appropriate using StandardScaler. Finally, we split the data into training (70%), validation (10%), and test (20%) sets for model development. 

One major limitation of the data is class imbalance, as there are significantly more individuals who voted than those who did not. This imbalance affects both model training and evaluation. We observed consistently high accuracy across different hyperparameter settings, making accuracy a less informative metric. Therefore, we used the F1 score for validation and evaluation, as it better captures the balance between precision and recall in imbalanced classification problems.

![data9.png](plots/data9.png)

Additionally, several variables are reported in categories rather than exact values. For example:

- Family income is provided in ranges rather than precise values.
- Age is top-coded (80 and 85+).

These limitations reduce precision of our measurements.

If data were available, we would have liked to include variables related to transportation access, such as:

- Car ownership
- Access to public transportation

These factors could significantly affect physical access to polling locations. In the absence of such variables, we attempted to proxy access using metropolitan status and mobility disability.

### Model 1: Logistic Regression

We chose the logistic regression model to understand how our chosen features can explain someone's likelihood of voting on a probability level. We picked this to have more information beyond the binary classification, to hopefully bettter understand how different characteristics contibute to a person's choice to vote.

We tuned three hyperparameters for this model:
* L1 vs. L2 - We intended to use the L2 (Ridge) Regression as we expected multicollinearity and wanted to penalize the coefficients fairly. However, we still tested both in our validation to ensure the L2 regularizer resulted in a better model.
* C - A lower C provides stronger regularization, pushing coefficients closer to zero. So, we expected to have a quite small C, still we tested values from 0.0001 to 10.
* class weight (None vs. balanced) - In the scikit learn's `LogisticRegression` class, class_weight='balanced' can help with imbalanced dataset, which may bias predictions toward our majority class of voted. Therefore, we expected a balanced class weight to perform better for this model.

As mentioned before, we chose to use the F1 score to capture the balance between precision and recall, as our data is imbalanced.

Our model with the highest mean test score of 0.501 had the following hyperparamters: 
* penalty = L1
* C = 0.00542867543932385
* class_weight = balanced

![logistic_regression_hyperparams.png](plots/logistic_regression_hyperparams.png)

Based on the coefficients, education has a the largest impact on the likelihood of voting out of all of our features. Having only a high school education increases the log-odds of not voting (0.52), while having an education higher than undergraduate decreases the log-odds of not voting (-0.54). This suggests that higher education is associated with a greater likelihood of voting, as was expected based on the data exploration. Other features with larger coefficients include mobility limitation (increases odds of not voting) and family income (decreases odds of not voting).

In [1]:
import pandas as pd

df = pd.read_csv('plots/linear_model_coefficients.csv')
df

,features,coef
0,SEX_female,[-0.06482636]
1,SEX_male,[0.15653566]
2,RACE_asian,[0.19627559]
3,RACE_black,[-0.24121507]
4,RACE_indian_aleut_eskimo,[0.19623929]
5,RACE_others,[0.06850972]
6,RACE_white,[-0.12810023]
7,EDUC_college_grad,[0.11223647]
8,EDUC_hs_grad,[0.52802307]
9,EDUC_master_higher,[-0.54855024]


The confusion matrix for this model shows that, false negatives (957; predicting someone voted when they did not) are much lower compared to false positives (2968; predicting someone did not vote when they did). We aimed to minimize false negatives to ensure we are catching those who do not vote, for targeted voter engagement policy -- so, this model does a decent job of that. However, the high number of false positives suggests the model over predicts `not_voted`, likely due to the class imbalance in the data.

![logistic_regression_confusion_matrix.png](plots/logistic_regression_confusion_matrix.png)

### Model 2: Random Forest

Towards the non-linear model, we considered using Random Forest and kNN. We chose a random forest model because 
- It is relatively easy to interpret compared to other non-linear models, and it provides useful metrics such as feature importance, which help us understand the impact of each variable. 
- Additionally, random forests handle non-linear relationships effectively and perform well with mixed types of features, making them suitable for our dataset.
We did not choose kNN because 
- Our training data is imbalanced, with significantly more “voted” observations than “not voted” ones. This imbalance can bias K-NN predictions toward the majority class.

For the random forest execution, following hyper-parameters were used:
- No. of Trees: Based on graphs for OOB error and AUC, 500 random trees are chosen as it indicated that there is no improvement in accuracy if we increase the trees after 300. So, 500 is taken as a conservative option as random forest hasn't got over-fitting problem and a factor of safety can help better results on test data.

![random_forest_obb.png](plots/random_forest_obb.png)

![random_forest_performance.png](plots/random_forest_performance.png)

- Threshold for majority class: Threshold of labelling is taken as 20% as it appeared to have good effect on actual predictions compared to something like a 50% prediction of trees. This increased non voter recall from 34% to 69%.

![random_forest_threshold.png](plots/random_forest_threshold.png)

- Class weight: Since our dataset is skewed with 77% Voted and 23% Not-Voted, the random forest would naturally incline towards predicting everything as Voted so as to attain 77% accuracy. However, this will mean all the Non-Voters are predicted wrongly and that defeats the whole purpose of our exercise. So, class_weight was taken as "balanced" so that minority errors are penalized more.

#### Model performance

- F1=0.4458  
- Recall=0.6849  
- Prec=0.3305  
- FP Rate=0.4195  
- FN Rate=0.3151

#### Top features
Permutation importance uses the validation dataset to shuffle the column of features to convert the feature into noise effectively, thereby measuring fall in performance as importance of the feature for model. Based on permutation importance, these are the top two features: INCOME_PER_PERSON (0.0123), AGE (0.0065)

#### Learning
- Our model has only one extracted feature and it shows up in the important ones. This suggests that a well-constructed feature can capture meaningful information from the original variables. This is why the extracted feature was identified as highly important by the random forest model.
- Hyper-paramter 'Class weight = Balanced' did not take care of the imbalance in random forest. It could help in constructing the tree. However, when we chose the threshold value of 0.20 for prediction, it was mimicking the original data imbalance. So, data imbalance had to be dealt in two stages- one with the class weight and other with threshold.

#### Limitations
Random Forest model used dummy variables to preserve consistency in model inputs across all models we ran in the project. This self-imposed constraint could have diluted the importance of categorical features, as they are treated as separate features by random forest trees.

### Model 3: Neural Networks

Neural networks are capable of learning complex, nonlinear relationships in data. Our goal is to predict whether an individual voted based on demographic characteristics, and neural networks are well-suited for capturing such patterns.

To train the model, we used the PyTorch library. The training process consists of the following steps: (1) for each training batch, perform a forward pass to compute predictions, (2) calculate the loss by comparing the predicted values (y_hat) with the true labels (y), (3) update model parameters using backpropagation, and (4) repeat this process over multiple epochs.

Because the dataset is imbalanced, we applied class weighting to give more importance to the minority class (not_voted). Specifically, we used the following formulation:

In [ ]:
self.pos_weight = torch.tensor([(self.torch_dataset.y_train == 0).sum()/(self.torch_dataset.y_train == 1).sum()]).float()

This increases the penalty for misclassifying individuals who did not vote, which aligns with the focus of our analysis.

We also observed that some models were unstable. Models with high variability in validation F1 scores (standard deviation greater than 0.1) were excluded to ensure stable performance:

In [ ]:
if f1_std > 0.1:
    result["f1"] = 0
    results.append(result)
    continue

Moreover, we used a stability-adjusted score (mean F1 minus standard deviation) to select the best model, ensuring both high performance and consistent behavior across training epochs..

In [ ]:
score = current_f1_mean - f1_std

In addition, models were excluded if their training loss failed to decrease or increased after several epochs, indicating poor convergence:

In [ ]:
elif epoch > 5 and avg_loss > initial_loss:
    return []

We tuned five key hyperparameters:

- Batch Size: To improve training efficiency compared to pure stochastic gradient descent (SGD), we used mini-batch training with batch sizes of 50, 100, and 200. For validation and testing, predictions were made without batching.

- Number of Hidden Layers: This determines the depth of the network. We tested models with 2, 5, and 10 hidden layers. While deeper networks can capture more complex patterns, they are also more prone to overfitting and instability.

- Hidden Dimension: This refers to the number of neurons in each hidden layer. Since the input dimension was 31 (after encoding), we tested 31 and 62. Larger dimensions increase model complexity.

- Learning Rate: This controls how quickly the model updates its parameters. A learning rate that is too high may cause divergence, while a rate that is too low may slow learning. We tested 0.01, 0.001, and 0.0001.

- Penalty Term: This represents the strength of L2 regularization. Strong regularization may lead to underfitting, while weak regularization may cause overfitting. We tested values of 0.01, 0.001, and 0.0001.

After running the hyperparameter search, we identified the best-performing model based on the validation F1 score. Hyperparameter selection was based on the validation set rather than the training set.

- Best Model: Batch size = 50, Number of hidden layers = 2, Hidden dimension = 31, Learning rate = 0.0001, Penalty rate = 0.001

As can be seen on the heatmap below, we found that models with a smaller number of hidden layers performed better, suggesting that simpler architectures were sufficient for this task and helped reduce overfitting.

![heatmap_hl_hd](plots/neural_networks_heatmap_hl_hd.png)

![f1_hidden_layers](plots/neural_networks_f1_hidden_layers.png)

The training curves show that performance quickly stabilizes and fluctuates slightly across epochs, suggesting limited improvement from additional training and reinforcing the importance of stability in model selection.

Deeper architectures showed signs of overfitting, which further supports the choice of a simpler model. (Hidden layer with 10 is not shown in the graph, as these models were highly unstable and were excluded during training.)

Using the best model, we evaluated performance on the test set. The number of false positives (predicting not_voted when the individual actually voted) and false negatives (predicting voted when the individual actually did not vote) were similar in magnitude, indicating that the model achieved balanced performance across classes.

- F1: 0.508605188800411 

- False Negative Rate: 0.3158258465791292 

- False Positive Rate: 0.304220643543669 

### Surprises

- Logistic Regression: Imbalanced data had large impact on correct predictions, despite efforts to mitigate differences. 

- Random Forest: The threshold for majority class had to be lowered to as low as 20% compared to baseline standard of 50%. While the threshold appeared low, it is probably aligned with the imbalance in data a non-voters constituted just 23% of overall cleaned data. Highest F1 occuring at 20% threshold was unexpected. Additionally, our only extracted feature performed the best among all.

- Neural Networks: Increasing neural network complexity does not always improve performance. The simplest model performed best. Additionally, Hyperparameters are important for performance & stability. Class imbalance in the target variable can be challenging.

### Conclusion
What did you learn about your research question? What did you learn about machine learning through the tasks?  How would you continue this research if you wanted to further investigate? If you had more time and infinite resources, how would you continue your project?

Based on the F1 scores, the the worst performing model was the random forest. This could have been because random forests have a hard time with imbalanced data, since they tend to vote for the majority class. The best performing model was the neural network, which may be due to its ability to learn complex, non-linear relationships between data.

Through this project, we've learned that the chioce of scorer is very important to the interpretation of the model. Originally, we chose accuracy, but later learned that accuracy does not work well for imbalanced data. Then, we chose precision -- but we realized that we'd flipped our outcomes (voted = 1, rather than voted = 0), and precision didn't make sense as our main method of evaluation, since false positives were no longer our focus. Finally, we settled on the F1 score, since it is sensitive to how well the model handles the minority class, and it provides a balance between precision and recall.

As for our research question, we learned that it's a bit more difficult to predict that a person will not vote, especially compared to predicting that a person will vote. This is likely due the imabalance of our data, meaning the models had more opportunities to learn what contributes to the odds of a person voting. Additionally, there may be many more factors that go into a choice not to vote that were not captured in our data such as access to reliable transportation and number of hours worked per day.

Therefore, to further this research, we could try to find individual-level voting data that included information about transportation and more details about people's jobs and home responsibilities. We originally wanted to predict a person's method of voting, so with more time that is something that we might explore.
